In [1]:
## 12/26更新：logistics_confirmed = pd.read_excel('./PLS和EPS物流方式纠偏清单_武汉&襄阳_数据不停变化新增直接更新_20251031.xlsx',sheet_name='Sheet1')
## 后续读取正确物流方式核对表带上日期(每月或每两个月末更新一次)

## 待更新项： 可能会有重复项，建议只匹配"PLS和EPS物流方式纠偏清单_武汉&襄阳_数据不停变化新增直接更新_20251031.xlsx"最后一行匹配的记录

In [2]:
import datetime
import pandas as pd
import datetime

In [3]:
# end_date = pd.to_datetime('2026-03-20')
# start_date = pd.to_datetime(end_date - datetime.timedelta(days=14))
end_date = pd.to_datetime(datetime.datetime.today().date())
start_date = pd.to_datetime(datetime.datetime.today().date() - datetime.timedelta(days=14))

year = datetime.datetime.today().year
month = datetime.datetime.today().month
day = datetime.datetime.today().day
today = year*10000 + month*100 + day

print(f'今日是：{today}')
print(f'start_date：{start_date}')
print(f'end_date：{end_date}')

今日是：20260515
start_date：2026-05-01 00:00:00
end_date：2026-05-15 00:00:00


## 需求：近双周(或其他周期),新增生效的零件物流方式是否正确

In [4]:
mass =  pd.read_excel('./eps3_mass.xlsx')

c:\Users\caicw\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


### 1、数据清洗：
1.已生效价格通知书号，2.生效日期/失效日期 只提取日期并转换成日期格式(后续需要比较)

In [5]:
mass = mass[mass['状态'].isin(['已发布','已确认','已审批'])]

mass['生效日期'] = pd.to_datetime(mass['生效日期'].str[:10])
mass['失效日期'] = pd.to_datetime(mass['失效日期'].str[:10])
mass['创建时间'] = pd.to_datetime(mass['创建时间'].str[:10])
mass['审批通过时间'] = pd.to_datetime(mass['审批通过时间'].str[:10])

In [6]:
usefull_mass_column = ['价格通知书号','供应商编码','供应商名称','零件号','零件名称','物流方式','物流费（元）','生效日期','失效日期','地区','创建时间','创建人','审批通过时间']

In [7]:
# 如果想重新运转，可以从此行开始
df_mass_useful = mass[usefull_mass_column]

#### 2、导出本期生效的价格零件及价格通知书

In [8]:
df_mass_useful_currently = df_mass_useful[(df_mass_useful['审批通过时间']<=end_date)&(df_mass_useful['审批通过时间']>=start_date)]
df_mass_useful_currently.head(2)

,价格通知书号,供应商编码,供应商名称,零件号,零件名称,物流方式,物流费（元）,生效日期,失效日期,地区,创建时间,创建人,审批通过时间
46,CJ2605140037,200258,威海万丰奥威汽轮有限公司,E3000617HR,车轮-金属本色,自送,8.0,2026-01-01,2026-12-31,Wuhan/,2026-05-14,张锐,2026-05-14
47,CJ2605140037,200258,威海万丰奥威汽轮有限公司,E300061766,车轮,自送,8.0,2026-01-01,2026-12-31,Wuhan/,2026-05-14,张锐,2026-05-14


In [9]:
print(f'当期新增零件数{df_mass_useful_currently.shape[0]}')

当期新增零件数1422


### 3、获取正确的物流方式

In [10]:
logistics_confirmed = pd.read_excel('./PLS和EPS物流方式纠偏清单_武汉&襄阳_数据不停变化新增直接更新_20251130.xlsx',sheet_name='Sheet1')
logistics_confirmed.head(2)

,工厂编码,物料编码,物料名称,供应商编码,供应商名称,EPS3正确物流方式,SCM承运商名称,承运商编码,返空承运商编码,返空承运商名称,EPS下传PLS物流方式,变更履历,重复
0,3100.0,7552160880,传动轴轴承螺栓,400034,中南橡胶集团中字橡胶汽车配件有限责任公司,NaN,末端风神物流调达,A5,A5,末端风神物流调达,NaN,NaN,31007552160880400034
1,1100.0,7552160880,传动轴轴承螺栓,400034,中南橡胶集团中字橡胶汽车配件有限责任公司,NaN,末端风神物流调达,A5,A5,末端风神物流调达,NaN,NaN,11007552160880400034


### 4、新增生效物料匹配正确的物流方式

In [11]:
df_mass_useful_currently.loc[:,'零件号'] = df_mass_useful_currently['零件号'].astype('str')
df_mass_useful_currently.loc[:,'供应商编码'] = df_mass_useful_currently['供应商编码'].astype('str')
df_mass_useful_currently.loc[:,'供应商编码-零件号'] = df_mass_useful_currently['供应商编码'] + df_mass_useful_currently['零件号']
df_mass_useful_currently.head(2)

C:\Users\caicw\AppData\Local\Temp\ipykernel_6020\1961222473.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mass_useful_currently.loc[:,'供应商编码-零件号'] = df_mass_useful_currently['供应商编码'] + df_mass_useful_currently['零件号']


,价格通知书号,供应商编码,供应商名称,零件号,零件名称,物流方式,物流费（元）,生效日期,失效日期,地区,创建时间,创建人,审批通过时间,供应商编码-零件号
46,CJ2605140037,200258,威海万丰奥威汽轮有限公司,E3000617HR,车轮-金属本色,自送,8.0,2026-01-01,2026-12-31,Wuhan/,2026-05-14,张锐,2026-05-14,200258E3000617HR
47,CJ2605140037,200258,威海万丰奥威汽轮有限公司,E300061766,车轮,自送,8.0,2026-01-01,2026-12-31,Wuhan/,2026-05-14,张锐,2026-05-14,200258E300061766


In [12]:
logistics_confirmed.loc[:,'物料编码'] = logistics_confirmed['物料编码'].astype('str')
logistics_confirmed.loc[:,'供应商编码'] = logistics_confirmed['供应商编码'].astype('str')
logistics_confirmed.loc[:,'供应商编码-物料编码'] = logistics_confirmed['供应商编码'] + logistics_confirmed['物料编码']
logistics_confirmed.head(2)

C:\Users\caicw\AppData\Local\Temp\ipykernel_6020\727808208.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['400034' '400034' '400034' ... '901555' '901555' '901555']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  logistics_confirmed.loc[:,'供应商编码'] = logistics_confirmed['供应商编码'].astype('str')


,工厂编码,物料编码,物料名称,供应商编码,供应商名称,EPS3正确物流方式,SCM承运商名称,承运商编码,返空承运商编码,返空承运商名称,EPS下传PLS物流方式,变更履历,重复,供应商编码-物料编码
0,3100.0,7552160880,传动轴轴承螺栓,400034,中南橡胶集团中字橡胶汽车配件有限责任公司,NaN,末端风神物流调达,A5,A5,末端风神物流调达,NaN,NaN,31007552160880400034,4000347552160880
1,1100.0,7552160880,传动轴轴承螺栓,400034,中南橡胶集团中字橡胶汽车配件有限责任公司,NaN,末端风神物流调达,A5,A5,末端风神物流调达,NaN,NaN,11007552160880400034,4000347552160880


In [13]:
df_mass_useful_currently_wuhan = df_mass_useful_currently[df_mass_useful_currently['地区'].isin(['Wuhan/','Wuhan/Xiangyang/', 'Xiangyang/Wuhan/'])]
df_mass_useful_currently_xiangyang = df_mass_useful_currently[df_mass_useful_currently['地区'].isin(['Xiangyang/', 'Wuhan/Xiangyang/', 'Xiangyang/Wuhan/'])]

In [14]:
df_mass_useful_currently_wuhan_with_correct_log = (
    df_mass_useful_currently_wuhan
    .merge(
        logistics_confirmed[logistics_confirmed['工厂编码'].isin([1100,3100,2100])],
        how='left',left_on='供应商编码-零件号',right_on='供应商编码-物料编码'
        )
    .dropna(subset=['EPS3正确物流方式'])
)
df_mass_useful_currently_wuhan_with_correct_log['是否一致'] = (
    df_mass_useful_currently_wuhan_with_correct_log.apply(
    lambda x: x['物流方式'] == x['EPS3正确物流方式'],
    axis=1
)
)

In [15]:
df_mass_useful_currently_wuhan_log_different = df_mass_useful_currently_wuhan_with_correct_log[df_mass_useful_currently_wuhan_with_correct_log['是否一致']!=True]

In [16]:
df_mass_useful_currently_wuhan_log_different.rename(
    columns={'供应商编码_x':'供应商编码','供应商名称_x':'供应商名称'},
    inplace=True
)

C:\Users\caicw\AppData\Local\Temp\ipykernel_6020\2551647519.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mass_useful_currently_wuhan_log_different.rename(


In [17]:
logic_confirmed_xiangyang = logistics_confirmed[logistics_confirmed['工厂编码'].isin([9190])].drop_duplicates(subset=['供应商编码-物料编码'],keep='first')
logic_confirmed_xiangyang.head(1)

,工厂编码,物料编码,物料名称,供应商编码,供应商名称,EPS3正确物流方式,SCM承运商名称,承运商编码,返空承运商编码,返空承运商名称,EPS下传PLS物流方式,变更履历,重复,供应商编码-物料编码
82,9190.0,9613251880,内六角花盘头自攻螺钉和平垫圈组合件,200054,上海东风汽车专用件有限公司,自送,襄阳丽晶斌,A1,NaN,NaN,NaN,NaN,91909613251880200054,2000549613251880


In [18]:
df_mass_useful_currently_xiangyang_with_correct_log = (
    df_mass_useful_currently_xiangyang
    .merge(
        logic_confirmed_xiangyang,
        how='left',left_on='供应商编码-零件号',right_on='供应商编码-物料编码'
        )
    .dropna(subset=['EPS3正确物流方式'])
)

df_mass_useful_currently_xiangyang_with_correct_log['是否一致'] = (
    df_mass_useful_currently_xiangyang_with_correct_log.apply(
    lambda x: x['物流方式'] == x['EPS3正确物流方式'],
    axis=1
)
)

In [19]:
df_mass_useful_currently_xiangyang_with_correct_log.head(1)

,价格通知书号,供应商编码_x,供应商名称_x,零件号,零件名称,物流方式,物流费（元）,生效日期,失效日期,地区,...,EPS3正确物流方式,SCM承运商名称,承运商编码,返空承运商编码,返空承运商名称,EPS下传PLS物流方式,变更履历,重复,供应商编码-物料编码,是否一致
2,CJ2605120028,100069,湖北东峻工贸有限公司,L100030460,制冷剂,自送,1.1,2026-01-01,2026-12-31,Wuhan/Xiangyang/,...,自送,供应商自送,A1,NaN,NaN,NaN,NaN,9190L100030460100069,100069L100030460,True


In [20]:
df_mass_useful_currently_xiangyang_with_correct_log = df_mass_useful_currently_xiangyang_with_correct_log[df_mass_useful_currently_xiangyang_with_correct_log['是否一致']!=True]

In [21]:
df_mass_useful_currently_xiangyang_with_correct_log.rename(
    columns={'供应商编码_x':'供应商编码','供应商名称_x':'供应商名称'},
    inplace=True
)

In [22]:
columns_used = ['价格通知书号', '供应商编码', '供应商名称', '零件号', '零件名称', '物流方式', '物流费（元）', '生效日期',
       '失效日期', '地区', '创建人', '审批通过时间', 'EPS3正确物流方式']

In [23]:
df_mass_useful_currently_wuhan_log_different = df_mass_useful_currently_wuhan_log_different[columns_used]
df_mass_useful_currently_wuhan_log_different

,价格通知书号,供应商编码,供应商名称,零件号,零件名称,物流方式,物流费（元）,生效日期,失效日期,地区,创建人,审批通过时间,EPS3正确物流方式
77,CJ2605120005,100044,武汉云鹤智博汽车零部件有限公司,K300289760,发动机右支架加强板,调达,0.45,2026-01-01,2026-12-31,Wuhan/,张锐,2026-05-13,自送
85,CJ2605120005,100044,武汉云鹤智博汽车零部件有限公司,K300289860,发动机右支架总成,调达,2.48,2026-01-01,2026-12-31,Wuhan/,张锐,2026-05-13,自送
284,CJ2605120005,100044,武汉云鹤智博汽车零部件有限公司,Z120012J-B0103,蓄电池固定杆,调达,0.01,2026-01-01,2026-12-31,Wuhan/,张锐,2026-05-13,自送
285,CJ2605120005,100044,武汉云鹤智博汽车零部件有限公司,Z120012J-B0103,蓄电池固定杆,调达,0.01,2026-01-01,2026-12-31,Wuhan/,张锐,2026-05-13,自送


In [25]:
df_mass_useful_currently_xiangyang_with_correct_log = df_mass_useful_currently_xiangyang_with_correct_log[columns_used]
df_mass_useful_currently_xiangyang_with_correct_log

,价格通知书号,供应商编码,供应商名称,零件号,零件名称,物流方式,物流费（元）,生效日期,失效日期,地区,创建人,审批通过时间,EPS3正确物流方式
50,CJ2605050005,770093,湖北诺御汽车零部件科技股份有限公司,Z700085060,显示屏外壳/支架（不可见）,调达,0.0,2026-01-01,2026-12-31,Xiangyang/,张锐,2026-05-08,自送
51,CJ2605050005,770093,湖北诺御汽车零部件科技股份有限公司,K300305060,线束安装支架Ⅱ,调达,0.0,2026-01-01,2026-12-31,Xiangyang/,张锐,2026-05-08,自送
52,CJ2605050005,770093,湖北诺御汽车零部件科技股份有限公司,K300406760,线束安装支架Ⅱ,调达,0.0,2026-01-01,2026-12-31,Xiangyang/,张锐,2026-05-08,自送
53,CJ2605050005,770093,湖北诺御汽车零部件科技股份有限公司,K300304660,制动踏板安装板,调达,0.0,2026-01-01,2026-12-31,Xiangyang/,张锐,2026-05-08,自送
54,CJ2605050005,770093,湖北诺御汽车零部件科技股份有限公司,ZA00016660,蓄电池电缆固定支架,调达,0.0,2026-01-01,2026-12-31,Xiangyang/,张锐,2026-05-08,自送
55,CJ2605050005,770093,湖北诺御汽车零部件科技股份有限公司,K200226060,远程控制器支架总成,调达,0.0,2026-01-01,2026-12-31,Xiangyang/,张锐,2026-05-08,自送
